# Program generator over retrieved facts (FinQANet-style stage 2)

**Research question.** Given the top 5 facts from the Notebook 10 retriever, how often does a fine-tuned
seq2seq model generate a FinQA program whose execution equals `exe_ans`? How much of the remaining error
comes from retrieval rather than generation?

**Protocol.** Settings come from `configs/generator_t5_base.toml`. The model trains on train records:
gold facts plus random same-record distractors (resampled each epoch), with the published `program` as
the target. Dev is evaluated under two conditions:

- **retrieved**: the top 5 facts from the verified Notebook 10 dev ranking file. This is the prediction
  system, and it reads no annotation field.
- **gold-facts oracle**: gold facts only. This is a diagnostic upper bound for the generator, not a
  prediction system.

Metrics: official execution accuracy (all 883 dev records), program accuracy (symbolic equivalence),
invalid-program rate and operand grounding. The checkpoint is chosen on dev execution accuracy under the
retrieved condition. The official test split is not loaded.

**Runtime.** A CUDA GPU is required (`docs/gpu-setup.md`). `FINEVIDENCE_SMOKE_TEST=1` runs a tiny CPU
check on a few records with a tiny random model; its outputs are not results.

In [1]:
from __future__ import annotations

import os

# Must be set before CUDA initializes; reduces fragmentation-related out-of-memory errors.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import json
import platform
import random
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, get_linear_schedule_with_warmup

from finevidence.data import load_finqa_split, parse_finqa_program
from finevidence.data.acquisition import download_finqa_splits
from finevidence.evaluation import gold_recall_at_k
from finevidence.generation import (
    build_oracle_examples,
    build_retrieved_examples,
    build_training_examples,
    load_generator_config,
    load_rankings,
    score_program_prediction,
    summarize_scores,
    verify_rankings_file,
)

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
SMOKE_TEST = os.environ.get("FINEVIDENCE_SMOKE_TEST") == "1"
config = load_generator_config(PROJECT_ROOT / "configs" / "generator_t5_base.toml")

pretrained_name = (
    "hf-internal-testing/tiny-random-T5ForConditionalGeneration"
    if SMOKE_TEST
    else config.pretrained_name
)
epochs = 1 if SMOKE_TEST else config.epochs
num_beams = 1 if SMOKE_TEST else config.num_beams
suffix = "-smoke" if SMOKE_TEST else ""
model_dir = Path(str(config.model_dir) + suffix)
model_dir = model_dir if model_dir.is_absolute() else PROJECT_ROOT / model_dir
predictions_dir = PROJECT_ROOT / config.predictions_dir
rankings_path = config.rankings_path
rankings_path = rankings_path if rankings_path.is_absolute() else PROJECT_ROOT / rankings_path

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not SMOKE_TEST and device.type != "cuda":
    raise RuntimeError(
        "Full training needs a CUDA GPU; set FINEVIDENCE_SMOKE_TEST=1 for a CPU check."
    )
use_amp = config.mixed_precision and device.type == "cuda"

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=PROJECT_ROOT
).stdout.strip()
environment = {
    "commit": commit,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "smoke_test": SMOKE_TEST,
    "pretrained_name": pretrained_name,
    "epochs": epochs,
    "num_beams": num_beams,
    "batch_size": config.batch_size,
    "micro_batch_size": config.micro_batch_size,
    "eval_batch_size": config.eval_batch_size,
    "mixed_precision": use_amp,
    "seed": config.seed,
}
print(json.dumps(environment, indent=2))

{
  "commit": "5f10139f0cdf75c87cae22f792ac66a2b65791c1",
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "transformers": "5.16.1",
  "device": "Tesla T4",
  "smoke_test": false,
  "pretrained_name": "t5-base",
  "epochs": 10,
  "num_beams": 4,
  "batch_size": 16,
  "micro_batch_size": 8,
  "eval_batch_size": 16,
  "mixed_precision": false,
  "seed": 42
}


## 1. Verified data, rankings, and generator inputs

The rankings file must match the SHA-256 recorded in the config, so it is the exact Notebook 10 output.
Training examples are rebuilt each epoch with a new distractor sample. Facts are rendered from the raw
context by key in every condition.

In [2]:
raw_dir = PROJECT_ROOT / config.dataset_root
print(download_finqa_splits(raw_dir, (config.train_split, config.evaluation_split)))
verify_rankings_file(rankings_path, config.rankings_sha256)
rankings = load_rankings(rankings_path)
train = load_finqa_split(raw_dir / f"{config.train_split}.json")
dev = load_finqa_split(raw_dir / f"{config.evaluation_split}.json")
if SMOKE_TEST:
    train, dev = train[:40], dev[:20]

retrieved_examples = build_retrieved_examples(dev, rankings, facts_per_input=config.facts_per_input)
oracle_examples = build_oracle_examples(dev)
first_epoch_examples = build_training_examples(
    train, facts_per_input=config.facts_per_input, seed=config.seed
)
print(
    {
        "train_records": len(train),
        "dev_records": len(dev),
        "test_loaded": False,
        "rankings_verified": str(rankings_path.name),
        "facts_per_input": config.facts_per_input,
    }
)
for name, example in [
    ("train", first_epoch_examples[0]),
    ("dev retrieved", retrieved_examples[0]),
    ("dev oracle", oracle_examples[0]),
]:
    print(
        f"\n[{name}] {example.record_id}\n  source: {example.source[:400]}\n  target: {example.target}"
    )

{'train': 'existing; verified, not overwritten', 'dev': 'existing; verified, not overwritten'}


{'train_records': 6251, 'dev_records': 883, 'test_loaded': False, 'rankings_verified': 'retriever_bert_base_dev_rankings.jsonl', 'facts_per_input': 5}

[train] ADI/2009/page_49.pdf-1
  source: question: what is the the interest expense in 2009? facts: [1] row: fair value of forward exchange contracts asset ( liability ) | october 31 2009: $ 6427 | november 1 2008: $ -23158 ( 23158 ) [2] . [3] if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million . [4] . [5] dollar-based exposures by entering into forward foreign currency exchange contracts .
  target: divide(100, 100), divide(3.8, #0)

[dev retrieved] V/2008/page_17.pdf-1
  source: question: what is the average payment volume per transaction for american express? facts: [1] row: american express | payments volume ( billions ): 637 | total volume ( billions ): 647 | total transactions ( billions ): 5.0 | cards ( millions ): 86 [2] row: visa inc. ( 1 ) | payments volume ( billions ): $ 2457 | to

## 2. Tokenized lengths

Sources longer than `max_source_length` are truncated from the end, which drops the lowest-ranked facts
first. Targets longer than `max_target_length` would be cut and could not be generated exactly.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_name)


def token_lengths(texts):
    return np.array([len(ids) for ids in tokenizer(list(texts)).input_ids])


length_rows = []
for name, texts, limit in [
    ("train source", [e.source for e in first_epoch_examples], config.max_source_length),
    ("train target", [e.target for e in first_epoch_examples], config.max_target_length),
    ("dev retrieved source", [e.source for e in retrieved_examples], config.max_source_length),
    ("dev oracle source", [e.source for e in oracle_examples], config.max_source_length),
]:
    lengths = token_lengths(texts)
    length_rows.append(
        {
            "input": name,
            "median": int(np.median(lengths)),
            "p95": int(np.percentile(lengths, 95)),
            "max": int(lengths.max()),
            "limit": limit,
            "over_limit": int((lengths > limit).sum()),
        }
    )
display(pd.DataFrame(length_rows))

,input,median,p95,max,limit,over_limit
0,train source,216,368,794,512,36
1,train target,11,26,50,64,0
2,dev retrieved source,212,402,704,512,14
3,dev oracle source,83,200,364,512,0


## 3. Train the generator

AdamW with linear warmup and decay. Each optimizer step uses `batch_size` examples, processed as
`micro_batch_size` chunks whose losses are weighted by their share of target tokens, so the accumulated
gradient equals the full-batch gradient. The decoder cache is disabled during training. Each epoch draws a new distractor sample and a new example order, both
seeded from the config seed and the epoch. After every epoch the model decodes the retrieved dev inputs
with beam search, and the checkpoint with the best dev execution accuracy is kept. Decoding uses
`clean_up_tokenization_spaces=False`, because the default clean-up changes row labels such as `u.k .`.

In [4]:
def clock(seconds: float) -> str:
    minutes, seconds = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h{minutes:02d}m{seconds:02d}s" if hours else f"{minutes}m{seconds:02d}s"


model = AutoModelForSeq2SeqLM.from_pretrained(pretrained_name).to(device)


def encode_sources(examples):
    return tokenizer(
        [example.source for example in examples],
        max_length=config.max_source_length,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)


@torch.no_grad()
def generate(examples, label: str) -> list[str]:
    model.eval()
    predictions, started = [], time.time()
    batch_size = config.eval_batch_size
    report_every = max(1, len(examples) // (batch_size * 4)) * batch_size
    for start in range(0, len(examples), batch_size):
        batch = examples[start : start + batch_size]
        with torch.autocast(device.type, enabled=use_amp):
            output = model.generate(
                **encode_sources(batch),
                max_new_tokens=config.max_target_length,
                num_beams=num_beams,
            )
        predictions.extend(
            tokenizer.batch_decode(
                output, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )
        )
        done = min(start + batch_size, len(examples))
        if done % report_every == 0 or done == len(examples):
            print(
                f"  {label}: decoded {done}/{len(examples)} ({clock(time.time() - started)})",
                flush=True,
            )
    return predictions


def evaluate(examples, predictions):
    scores = [
        score_program_prediction(record, prediction, example.fact_texts)
        for record, example, prediction in zip(dev, examples, predictions, strict=True)
    ]
    return scores, summarize_scores(scores)


steps_per_epoch = -(-len(train) // config.batch_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(config.warmup_ratio * steps_per_epoch * epochs), steps_per_epoch * epochs
)
scaler = torch.amp.GradScaler(enabled=use_amp)
history, best_accuracy = [], -1.0
model_dir.mkdir(parents=True, exist_ok=True)
report_every = max(1, steps_per_epoch // 10)
run_started = time.time()
print(
    f"Training {epochs} epoch(s) x {steps_per_epoch} steps, batch {config.batch_size}, "
    f"beams {num_beams}, on {environment['device']}",
    flush=True,
)

for epoch in range(epochs):
    model.train()
    epoch_examples = build_training_examples(
        train, facts_per_input=config.facts_per_input, seed=config.seed + epoch
    )
    random.Random(config.seed * 1_000_003 + epoch).shuffle(epoch_examples)
    started, losses = time.time(), []
    print(f"\nEpoch {epoch + 1}/{epochs}", flush=True)
    for step, start in enumerate(range(0, len(epoch_examples), config.batch_size), start=1):
        batch = epoch_examples[start : start + config.batch_size]
        chunks = [
            batch[i : i + config.micro_batch_size]
            for i in range(0, len(batch), config.micro_batch_size)
        ]
        chunk_labels = []
        for chunk in chunks:
            labels = tokenizer(
                [example.target for example in chunk],
                max_length=config.max_target_length,
                truncation=True,
                padding=True,
                return_tensors="pt",
            ).input_ids.to(device)
            labels[labels == tokenizer.pad_token_id] = -100
            chunk_labels.append(labels)
        batch_tokens = sum(int((labels != -100).sum()) for labels in chunk_labels)
        optimizer.zero_grad(set_to_none=True)
        batch_loss = 0.0
        for chunk, labels in zip(chunks, chunk_labels, strict=True):
            with torch.autocast(device.type, enabled=use_amp):
                loss = model(**encode_sources(chunk), labels=labels, use_cache=False).loss
            # Token-weighted so the summed gradient equals the full-batch mean-token gradient.
            weighted = loss * int((labels != -100).sum()) / batch_tokens
            scaler.scale(weighted).backward()
            batch_loss += weighted.item()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(batch_loss)
        if step % report_every == 0 or step == steps_per_epoch:
            elapsed = time.time() - started
            print(
                f"  step {step}/{steps_per_epoch} ({100 * step / steps_per_epoch:.0f}%)"
                f" | loss {np.mean(losses[-report_every:]):.4f} | elapsed {clock(elapsed)}"
                f" | epoch time left ~{clock(elapsed / step * (steps_per_epoch - step))}",
                flush=True,
            )
    train_seconds = time.time() - started
    if device.type == "cuda":
        peak_memory_gb = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
        print(f"  peak GPU memory so far: {peak_memory_gb} GB", flush=True)
    _, summary = evaluate(
        retrieved_examples, generate(retrieved_examples, f"epoch {epoch + 1} dev")
    )
    history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "train_seconds": round(train_seconds, 1),
            "epoch_seconds_with_dev": round(time.time() - started, 1),
            **{f"dev_retrieved_{key}": value for key, value in summary.items() if key != "records"},
        }
    )
    improved = summary["execution_accuracy"] > best_accuracy
    if improved:
        best_accuracy = summary["execution_accuracy"]
        model.save_pretrained(model_dir / "best")
        tokenizer.save_pretrained(model_dir / "best")
    (model_dir / "history.json").write_text(
        json.dumps({"environment": environment, "history": history}, indent=2)
    )
    per_epoch = (time.time() - run_started) / (epoch + 1)
    print(
        f"Epoch {epoch + 1}/{epochs} done: {history[-1]}"
        f" | {'new best, checkpoint saved' if improved else 'not better than best'}"
        f" | total elapsed {clock(time.time() - run_started)}"
        f" | ~{clock(per_epoch * (epochs - epoch - 1))} left for {epochs - epoch - 1} epoch(s)",
        flush=True,
    )
display(pd.DataFrame(history))

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Training 10 epoch(s) x 391 steps, batch 16, beams 4, on Tesla T4



Epoch 1/10


  step 39/391 (10%) | loss 4.8383 | elapsed 0m51s | epoch time left ~7m42s


  step 78/391 (20%) | loss 1.8277 | elapsed 1m47s | epoch time left ~7m12s


...

  epoch 10 dev: decoded 624/883 (2m03s)


  epoch 10 dev: decoded 832/883 (2m42s)


  epoch 10 dev: decoded 883/883 (2m51s)


Epoch 10/10 done: {'epoch': 9, 'train_loss': 0.1323739556033555, 'train_seconds': 589.8, 'epoch_seconds_with_dev': 765.9, 'dev_retrieved_execution_correct': 515, 'dev_retrieved_execution_accuracy': 0.583239, 'dev_retrieved_program_accuracy': 0.552661, 'dev_retrieved_invalid_rate': 0.010193, 'dev_retrieved_operand_grounding_rate': 0.97205} | not better than best | total elapsed 2h07m51s | ~0m00s left for 0 epoch(s)


,epoch,train_loss,train_seconds,epoch_seconds_with_dev,dev_retrieved_execution_correct,dev_retrieved_execution_accuracy,dev_retrieved_program_accuracy,dev_retrieved_invalid_rate,dev_retrieved_operand_grounding_rate
0,0,1.341749,574.8,759.4,299,0.338618,0.308041,0.016988,0.970370
1,1,0.559909,584.5,784.4,373,0.422424,0.382786,0.005663,0.964666
2,2,0.427195,581.1,784.2,437,0.494904,0.462061,0.007928,0.962924
3,3,0.350939,580.1,759.6,455,0.515289,0.477916,0.005663,0.963158
4,4,0.288781,584.0,752.5,488,0.552661,0.513024,0.005663,0.970343
5,5,0.239716,579.0,767.6,494,0.559456,0.519819,0.006795,0.963446
6,6,0.206430,582.0,772.4,508,0.575311,0.543601,0.009060,0.973792
7,7,0.173068,581.9,755.6,501,0.567384,0.535674,0.006795,0.969120
8,8,0.153101,579.7,749.6,523,0.592299,0.556059,0.007928,0.970436
9,9,0.132374,589.8,765.9,515,0.583239,0.552661,0.010193,0.972050


## 4. Selected checkpoint: retrieved inputs vs gold-facts oracle

Both conditions use the same checkpoint and decoding. The reference row is the best earlier dev run
(Notebook 08 conservative arithmetic, re-scored in Notebook 09 under execution accuracy).

In [5]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir / "best").to(device)
results, all_scores = {}, {}
for condition, examples in [
    ("retrieved", retrieved_examples),
    ("gold_facts_oracle", oracle_examples),
]:
    predictions = generate(examples, f"selected checkpoint, {condition}")
    scores, summary = evaluate(examples, predictions)
    results[condition], all_scores[condition] = summary, scores
    path = predictions_dir / f"{config.experiment_name}{suffix}_dev_{condition}_predictions.jsonl"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for example, score in zip(examples, scores, strict=True):
            handle.write(
                json.dumps({**vars(score), "fact_keys": example.fact_keys, "commit": commit}) + "\n"
            )
    print(f"  wrote {path.relative_to(PROJECT_ROOT)}", flush=True)

summary_table = pd.DataFrame(results).T
summary_table.loc["reference: notebook 08 arithmetic"] = {
    "records": 883,
    "execution_correct": 35,
    "execution_accuracy": 0.039638,
}
display(summary_table)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

  selected checkpoint, retrieved: decoded 208/883 (0m34s)


  selected checkpoint, retrieved: decoded 416/883 (1m16s)


  selected checkpoint, retrieved: decoded 624/883 (1m59s)


  selected checkpoint, retrieved: decoded 832/883 (2m36s)


...

  selected checkpoint, gold_facts_oracle: decoded 624/883 (1m11s)


  selected checkpoint, gold_facts_oracle: decoded 832/883 (1m34s)


  selected checkpoint, gold_facts_oracle: decoded 883/883 (1m40s)


  wrote data/processed/generator-t5-base-dev-v1_dev_gold_facts_oracle_predictions.jsonl


,records,execution_correct,execution_accuracy,program_accuracy,invalid_rate,operand_grounding_rate
retrieved,883.0,523.0,0.592299,0.556059,0.007928,0.970436
gold_facts_oracle,883.0,578.0,0.654587,0.618347,0.007928,0.975546
reference: notebook 08 arithmetic,883.0,35.0,0.039638,NaN,NaN,NaN


## 5. Post-hoc breakdown (dev annotations read only after predictions)

Execution accuracy by program length, by number of gold facts, and, for the retrieved condition, by
whether every gold fact was in the top 5. These are diagnostics, not causal error labels.

In [6]:
rows = []
for record, retrieved_score, oracle_score in zip(
    dev, all_scores["retrieved"], all_scores["gold_facts_oracle"], strict=True
):
    steps = len(parse_finqa_program(record["qa"]["program"]).steps)
    gold = list(record["qa"]["gold_inds"])
    rows.append(
        {
            "program_steps": "1" if steps == 1 else "2" if steps == 2 else "3+",
            "gold_facts": "1" if len(gold) == 1 else "2+",
            "all_gold_in_top_k": gold_recall_at_k(
                rankings[record["id"]], gold, config.facts_per_input
            ).all_gold,
            "retrieved_correct": retrieved_score.execution_correct,
            "oracle_correct": oracle_score.execution_correct,
        }
    )
breakdown = pd.DataFrame(rows)
for column in ["program_steps", "gold_facts", "all_gold_in_top_k"]:
    display(
        breakdown.groupby(column)
        .agg(
            records=("retrieved_correct", "size"),
            retrieved_accuracy=("retrieved_correct", "mean"),
            oracle_accuracy=("oracle_correct", "mean"),
        )
        .round(4)
    )

,records,retrieved_accuracy,oracle_accuracy
program_steps,,,
1,523,0.6558,0.7208
2,287,0.5540,0.6341
3+,73,0.2877,0.2603


,records,retrieved_accuracy,oracle_accuracy
gold_facts,,,
1,416,0.6274,0.6947
2+,467,0.5610,0.6188


,records,retrieved_accuracy,oracle_accuracy
all_gold_in_top_k,,,
False,115,0.2783,0.3652
True,768,0.6393,0.6979


## Interpretation

Run: Kaggle Tesla T4, commit `5f10139`, torch 2.11.0+cu128, transformers 5.16.1, seed 42. `t5-base` in fp32, effective batch 16 (2 × 8 token-weighted accumulation), beam size 4. Peak GPU memory was 9.55 GB. Each epoch took about 580 s of training plus about 3 min of dev decoding; the 10 epochs took 2.1 h. The outputs above are from that run. Re-scoring both saved prediction files locally reproduced every metric and breakdown, and the downloaded checkpoint regenerated the first 16 retrieved predictions exactly.

- **Main result.** With the Notebook 10 retriever's top 5 facts, dev execution accuracy is **523/883 (0.5923; normal-approximation 95% interval about 0.560–0.625)** and program accuracy is 0.5561. The best earlier dev run (Notebook 08 conservative arithmetic) reached 35/883 (0.0396) under the same metric.
- **Generated programs are well formed and grounded.** 0.79% fail to parse or execute, and 97.0% of their numeric literals occur in the input facts. 32 retrieved-condition answers are execution-correct without being symbolically equivalent to the gold program.
- **The generator is the larger remaining bottleneck.** With gold facts only, accuracy is 578/883 (0.6546). Perfect retrieval would therefore add about 6 points, while about 35% of questions stay wrong even with perfect evidence. Per record: 494 are correct in both conditions, 84 only with gold facts, and 29 only with retrieved facts.
- **Distractors cost accuracy even when evidence is complete.** On the 768 records whose gold facts are all in the top 5, accuracy is 0.6393 with retrieved inputs vs 0.6979 with gold facts only. The 115 records missing a gold fact still reach 0.2783.
- **Multi-step programs are the hardest case.** By gold program length, accuracy is 0.6558 for 1 step, 0.5540 for 2 steps and 0.2877 for 3+ steps (oracle: 0.7208, 0.6341, 0.2603). Records with several gold facts score 0.5610, against 0.6274 for single-fact records.
- **Training curve.** Dev execution accuracy rose from 0.3386 (epoch 1) to a best of 0.5923 at epoch 9 (selected), then 0.5832 at epoch 10. Training loss fell from 1.34 to 0.13.

**Limitations.** Epoch 9 was selected on dev and is also reported on dev; epochs 7–10 span about 2.5 points, so the figure is somewhat optimistic. Unbiased numbers require the held-out M4 evaluation. This is one seed, one GPU type, and fixed hyperparameters (learning rate, epochs and beam size are FinEvidence choices). The oracle uses about 2 facts per input while training used 5, so it is a different input distribution, not only "perfect retrieval". The breakdowns are post-hoc correlations, not causal error labels.

**Decision.** This retriever + generator pipeline is the M3 candidate system, compared with the lexical and arithmetic baselines under the same dev protocol and metric. M4 should start with categorized error analysis of the 360 retrieved-condition errors, focusing on multi-step programs and distractor confusion, before any further tuning.